In [16]:
#!pip install onnx
#!pip install onnxruntime

In [17]:
import torch
import torch.nn as nn
import time
import torch.onnx

import onnxruntime as ort
import numpy as np
import onnx

In [18]:

# Define a simple feedforward network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Define layers
        self.fc1 = nn.Linear(2, 2)  # Input to layer 1
        self.fc2 = nn.Linear(2, 3)  # Layer 1 to layer 2
        self.fc3 = nn.Linear(3, 4)  # Layer 2 to layer 3 (output)

    def forward(self, x):
        # Forward pass through the network
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = self.fc3(x)  # Output layer has no activation
        return x

# Instantiate the model
model = SimpleNet()

# Example input for inference
example_input = torch.tensor([[1.0, 2.0]])

# Perform inference
output = model(example_input)

# Print the inference output
print("PyTorch Inference Output:", output.detach().numpy())

# Export model to ONNX
torch.onnx.export(model, example_input, "simple_model.onnx")

PyTorch Inference Output: [[ 0.5402926   0.0199194  -0.6320982  -0.10534626]]


In [19]:

# Specify the path for the ONNX model file
onnx_model_path = "simple_model.onnx"

# Convert the PyTorch model to ONNX
torch.onnx.export(
    model,                          # model being exported
    example_input,                  # model input (or a tuple for multiple inputs)
    onnx_model_path,                # where to save the model (can be a file or file-like object)
    input_names=["input"],          # the model's input names
    output_names=["output"],        # the model's output names
)

print(f"Model successfully converted to ONNX: {onnx_model_path}")



Model successfully converted to ONNX: simple_model.onnx


In [20]:

# Load the ONNX model
onnx_model_path = "simple_model.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Prepare sample input data (same shape as the PyTorch model)
onnx_input = np.array([[1.0, 2.0]], dtype=np.float32)


# Run inference on the ONNX model
onnx_output = ort_session.run(None, {"input": onnx_input})

# Print the ONNX inference result
print("ONNX Inference Output:", onnx_output)

ONNX Inference Output: [array([[ 0.5402926 ,  0.0199194 , -0.6320982 , -0.10534627]],
      dtype=float32)]


In [21]:
model = onnx.load("simple_model.onnx")
onnx.checker.check_model(model)

# Print basic model info
print("Model Graph:")
print(model.graph)

# Iterate and print details of each node
for node in model.graph.node:
    print(f"Node Name: {node.name}")
    print(f"Inputs: {node.input}")
    print(f"Outputs: {node.output}")
    print("-----")
    


Model Graph:
node {
  input: "input"
  input: "fc1.weight"
  input: "fc1.bias"
  output: "/fc1/Gemm_output_0"
  name: "/fc1/Gemm"
  op_type: "Gemm"
  attribute {
    name: "alpha"
    type: FLOAT
    f: 1
  }
  attribute {
    name: "beta"
    type: FLOAT
    f: 1
  }
  attribute {
    name: "transB"
    type: INT
    i: 1
  }
}
node {
  input: "/fc1/Gemm_output_0"
  output: "/Sigmoid_output_0"
  name: "/Sigmoid"
  op_type: "Sigmoid"
}
node {
  input: "/Sigmoid_output_0"
  input: "fc2.weight"
  input: "fc2.bias"
  output: "/fc2/Gemm_output_0"
  name: "/fc2/Gemm"
  op_type: "Gemm"
  attribute {
    name: "alpha"
    type: FLOAT
    f: 1
  }
  attribute {
    name: "beta"
    type: FLOAT
    f: 1
  }
  attribute {
    name: "transB"
    type: INT
    i: 1
  }
}
node {
  input: "/fc2/Gemm_output_0"
  output: "/Sigmoid_1_output_0"
  name: "/Sigmoid_1"
  op_type: "Sigmoid"
}
node {
  input: "/Sigmoid_1_output_0"
  input: "fc3.weight"
  input: "fc3.bias"
  output: "output"
  name: "/fc3/Gemm

In [23]:
import netron

netron.start('simple_model.onnx')

Serving 'simple_model.onnx' at http://localhost:8080


('localhost', 8080)